In [ ]:
!pip install rich

In [ ]:
!pip install requests pandas numpy tqdm

In [ ]:
!pip install tabulate

In [ ]:
!pip install requests beautifulsoup4 pandas lxml html5lib tqdm

# test the api

In [ ]:
import requests

API_KEY = "naH0KGr02DcE5mjhEMhY7kjVUJQnx6gVbojoSfSg"
ENDPOINT = f"https://api.nal.usda.gov/fdc/v1/foods/search?api_key={API_KEY}"

def fetch_and_format_food_data(query):
    payload = {
        "query": query,
        "dataType": ["Foundation", "SR Legacy"],
        "pageSize": 1 # Just grabbing the top result for this example
    }

    response = requests.post(ENDPOINT, json=payload)
    data = response.json()

    if not data.get('foods'):
        return "No food found."

    food = data['foods'][0]

    # 1. Initialize our clean metadata dictionary
    metadata = {
        "fdc_id": food.get("fdcId"),
        "food_name": food.get("description"),
        "food_category": food.get("foodCategory", "Unknown"),
        "data_type": food.get("dataType"),
        "calories": 0.0,
        "protein_g": 0.0,
        "carbs_g": 0.0,
        "fat_g": 0.0
    }

    # 2. Extract the specific macros (USDA uses specific IDs or names for nutrients)
    # Protein = "Protein", Fat = "Total lipid (fat)", Carbs = "Carbohydrate, by difference", Calories = "Energy"
    for nutrient in food.get('foodNutrients', []):
        name = nutrient.get('nutrientName', '')
        value = float(nutrient.get('value', 0))

        if name == "Protein":
            metadata["protein_g"] = value
        elif name == "Total lipid (fat)":
            metadata["fat_g"] = value
        elif name == "Carbohydrate, by difference":
            metadata["carbs_g"] = value
        elif name == "Energy" and nutrient.get('unitName') == 'KCAL':
            metadata["calories"] = value

    # 3. Create the plain text chunk that the LLM will actually "read"
    text_chunk = f"100 grams of {metadata['food_name']} contains {metadata['calories']} calories, {metadata['protein_g']}g of protein, {metadata['carbs_g']}g of carbs, and {metadata['fat_g']}g of fat. It belongs to the {metadata['food_category']} category."

    # 4. The Final Pinecone Payload
    pinecone_payload = {
        "id": str(metadata["fdc_id"]),
        "metadata": {
            "text": text_chunk,
            **metadata # This unpacks all the categories we defined above
        }
    }

    return pinecone_payload

# Let's run it!
formatted_data = fetch_and_format_food_data("Raw Salmon")
print(formatted_data)

{'id': '173688', 'metadata': {'text': '100 grams of Fish, salmon, chinook, raw contains 179.0 calories, 19.9g of protein, 0.0g of carbs, and 10.4g of fat. It belongs to the Finfish and Shellfish Products category.', 'fdc_id': 173688, 'food_name': 'Fish, salmon, chinook, raw', 'food_category': 'Finfish and Shellfish Products', 'data_type': 'SR Legacy', 'calories': 179.0, 'protein_g': 19.9, 'carbs_g': 0.0, 'fat_g': 10.4}}


# fetch Structure food data from https://fdc.nal.usda.gov/

In [ ]:
egyptian_hybrid_fitness_foods = [
    # --- CORE PROTEINS (Meat & Poultry) ---
    "Chicken breast, raw", "Chicken thigh, raw", "Turkey breast, raw",
    "Ground beef, 95% lean, raw", "Beef, top sirloin, raw",
    "Beef, liver, raw", # Kebda (Very high iron/protein staple in Egypt)
    "Rabbit, raw", # Araneb (Very lean protein popular in Egypt)

    # --- SEAFOOD ---
    "Tilapia, raw", # Samak Bolti (The #1 fish in Egypt)
    "Salmon, raw", "Tuna, light, canned in water", "Shrimp, raw",
    "Mackerel, raw", "Sardines, canned",

    # --- EGGS & DAIRY (Local Staples) ---
    "Egg, whole, raw", "Egg white, raw",
    "Cheese, cottage, lowfat", # Closest USDA match to Gebna Areesh (Bodybuilding staple)
    "Cheese, feta", # Closest to Gebna Beda/Domiati
    "Yogurt, Greek, plain, nonfat", "Yogurt, plain, whole milk", # Zabadi
    "Milk, whole", "Milk, skim", "Kefir, plain", # Rayeb equivalent

    # --- PLANT-BASED PROTEINS & LEGUMES (The Egyptian Powerhouse) ---
    "Broadbeans (fava beans), raw", # Ful Medames! Essential.
    "Lupins, raw", # Termis! One of the best protein snacks in the world.
    "Lentils, raw", # Ads (For Lentil soup/Koshary)
    "Chickpeas, raw", # Hummus (For Koshary or dips)
    "Black beans, raw", "Tofu, firm, raw",

    # --- COMPLEX CARBOHYDRATES ---
    "Bread, pita, whole-wheat", # Closest USDA match for Eish Baladi
    "Rice, white, short-grain, raw", # Egyptian Rice is short-grain
    "Rice, brown, raw", "Oats, raw",
    "Sweet potato, raw", # Batata (Very popular street food/carb)
    "Potato, russet, raw", "Macaroni, dry", # For Koshary/Pasta dishes
    "Quinoa, raw", "Bulgur, dry", # Borghol

    # --- HEALTHY FATS ---
    "Tahini", # Tehina (Absolute staple healthy fat)
    "Olive oil, extra virgin", "Ghee", # Samna (Use sparingly but needed for local recipes)
    "Avocado, raw", "Almonds, unroasted", "Walnuts, raw", "Peanuts, raw",
    "Peanut butter, smooth", "Chia seeds, dried", "Flaxseed, whole",

    # --- VEGETABLES (Micronutrients & Volume) ---
    "Broccoli, raw", "Spinach, raw", "Cauliflower, raw",
    "Cabbage, raw", "Bell pepper, green, raw", "Bell pepper, red, raw",
    "Onion, raw", "Garlic, raw", "Zucchini, raw", "Cucumber, raw",
    "Tomatoes, red, ripe, raw", "Carrots, raw", "Eggplant, raw", # Betingan
    "Molokhia, raw", # Jute leaves (Might be hard to find in USDA, but worth trying)

    # --- FRUITS (Local Favorites) ---
    "Dates, medjool", # Tamr / Balah (Essential pre-workout in Egypt)
    "Guava, raw", # Gawafa
    "Pomegranate, raw", # Romman
    "Figs, raw", # Teen
    "Watermelon, raw", "Mango, raw", "Banana, raw",
    "Apple, raw", "Orange, raw", "Grapes, raw",

    # --- SUPPLEMENTS & EXTRAS ---
    "Whey protein isolate","Honey, raw",
    "Cocoa powder, unsweetened", "Cinnamon, ground", "Cumin, ground" # Kamoun
]

print(f"Loaded {len(egyptian_hybrid_fitness_foods)} Egyptian-Global hybrid ingredients!")

Loaded 76 Egyptian-Global hybrid ingredients!


In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm
import math

# ==========================================
# 1. SETUP YOUR USDA KEY
# ==========================================
USDA_API_KEY = "naH0KGr02DcE5mjhEMhY7kjVUJQnx6gVbojoSfSg"
USDA_ENDPOINT = f"https://api.nal.usda.gov/fdc/v1/foods/search?api_key={USDA_API_KEY}"


# ==========================================
# 3. THE MASTER FETCH LOOP (ENHANCED WITH TQDM & NaN)
# ==========================================
extracted_foods_data = []
failed_foods = []

print(f"Starting fetch pipeline for {len(egyptian_hybrid_fitness_foods)} foods...\n")

for food_item in tqdm(egyptian_hybrid_fitness_foods, desc="Fetching foods", unit="food"):
    payload = {"query": food_item, "dataType": ["Foundation", "SR Legacy"], "pageSize": 1}
    response = requests.post(USDA_ENDPOINT, json=payload)

    if response.status_code == 200 and response.json().get('foods'):
        food_data = response.json()['foods'][0]

        # Step B: Extract Metadata
        metadata = {
            "search_query": food_item,
            "fdc_id": food_data.get("fdcId"),
            "food_name": food_data.get("description"),
            "food_category": food_data.get("foodCategory", "Unknown"),
            "data_type": food_data.get("dataType"),

            # Initialize with 0.0 for found items
            "calories": 0.0, "protein_g": 0.0, "carbs_g": 0.0, "fat_g": 0.0,
            "fiber_g": 0.0, "sugar_g": 0.0, "saturated_fat_g": 0.0,
            "cholesterol_mg": 0.0, "sodium_mg": 0.0, "iron_mg": 0.0,
            "calcium_mg": 0.0, "potassium_mg": 0.0, "magnesium_mg": 0.0,
            "vitamin_c_mg": 0.0, "vitamin_b12_ug": 0.0
        }

        # Extract specific macros
        for nutrient in food_data.get('foodNutrients', []):
            name = nutrient.get('nutrientName', '')
            value = float(nutrient.get('value', 0))

            if name == "Protein": metadata["protein_g"] = value
            elif name == "Total lipid (fat)": metadata["fat_g"] = value
            elif name == "Carbohydrate, by difference": metadata["carbs_g"] = value
            elif name == "Energy" and nutrient.get('unitName') == 'KCAL': metadata["calories"] = value
            elif name == "Fiber, total dietary": metadata["fiber_g"] = value
            elif name == "Sugars, total including NLEA": metadata["sugar_g"] = value
            elif name == "Fatty acids, total saturated": metadata["saturated_fat_g"] = value
            elif name == "Cholesterol": metadata["cholesterol_mg"] = value
            elif name == "Sodium, Na": metadata["sodium_mg"] = value
            elif name == "Iron, Fe": metadata["iron_mg"] = value
            elif name == "Calcium, Ca": metadata["calcium_mg"] = value
            elif name == "Potassium, K": metadata["potassium_mg"] = value
            elif name == "Magnesium, Mg": metadata["magnesium_mg"] = value
            elif name == "Vitamin C, total ascorbic acid": metadata["vitamin_c_mg"] = value
            elif name == "Vitamin B-12": metadata["vitamin_b12_ug"] = value

        # Create the text chunk for successful fetches
        text_chunk = f"""
        Food: {metadata['food_name']}

        Per 100 grams:
        - Calories: {metadata['calories']} kcal
        - Protein: {metadata['protein_g']} g
        - Carbohydrates: {metadata['carbs_g']} g
        - Fat: {metadata['fat_g']} g
        - Fiber: {metadata['fiber_g']} g
        - Sugar: {metadata['sugar_g']} g
        - Saturated Fat: {metadata['saturated_fat_g']} g
        - Cholesterol: {metadata['cholesterol_mg']} mg
        - Sodium: {metadata['sodium_mg']} mg
        - Iron: {metadata['iron_mg']} mg
        - Calcium: {metadata['calcium_mg']} mg
        - Potassium: {metadata['potassium_mg']} mg
        - Magnesium: {metadata['magnesium_mg']} mg
        - Vitamin C: {metadata['vitamin_c_mg']} mg
        - Vitamin B12: {metadata['vitamin_b12_ug']} µg

        Category: {metadata['food_category']}
        Data Source Type: {metadata['data_type']}
        """
        metadata["text_chunk"] = text_chunk.strip()
        extracted_foods_data.append(metadata)

    else:
        # --- THE NEW FAILURE LOGIC ---
        tqdm.write(f"  ⚠️ USDA Error: Could not find {food_item}. Appending NaNs.")
        failed_foods.append(food_item)

        # Create a dictionary with NaN for all numeric values
        nan_metadata = {
            "search_query": food_item,
            "fdc_id": math.nan,
            "food_name": food_item, # Fallback to your query name
            "food_category": "Unknown",
            "data_type": "Imputed",
            "calories": math.nan, "protein_g": math.nan, "carbs_g": math.nan, "fat_g": math.nan,
            "fiber_g": math.nan, "sugar_g": math.nan, "saturated_fat_g": math.nan,
            "cholesterol_mg": math.nan, "sodium_mg": math.nan, "iron_mg": math.nan,
            "calcium_mg": math.nan, "potassium_mg": math.nan, "magnesium_mg": math.nan,
            "vitamin_c_mg": math.nan, "vitamin_b12_ug": math.nan,
            "text_chunk": f"Food: {food_item}\n\nNutritional data missing. Requires imputation."
        }
        extracted_foods_data.append(nan_metadata)

    time.sleep(1) # Respect API rate limits

# ==========================================
# 4. SAVE RESULTS AND LOG FAILURES
# ==========================================
print(f"\nFetch Complete! Processed {len(extracted_foods_data)} total rows.")
print(f"Failed to find {len(failed_foods)} items. They have been populated with NaNs.")

# Save the main dataset
df = pd.DataFrame(extracted_foods_data)
df.to_csv('egypt_foods_metadata.csv', index=False)
print("💾 Saved 'egypt_foods_metadata.csv'.")

# Save the failed items to a separate text file so you can review them easily
if failed_foods:
    with open('failed_foods_log.txt', 'w') as f:
        for item in failed_foods:
            f.write(f"{item}\n")
    print("📝 Saved 'failed_foods_log.txt'. Review this file to see which items need manual query adjustments.")

Starting fetch pipeline for 76 foods...



Fetching foods: 100%|██████████| 76/76 [01:37<00:00,  1.28s/food]


Fetch Complete! Processed 76 total rows.
Failed to find 0 items. They have been populated with NaNs.
💾 Saved 'egypt_foods_metadata.csv'.


In [ ]:
from tabulate import tabulate
import pandas as pd

def display_wide_table_transposed(df, n_rows=3):
    """
    Display wide DataFrame by transposing it - much more readable!
    Shows each row as a column with field names as rows
    """
    # Select rows to display
    sample_df = df.head(n_rows).T

    sample_df.columns = [f"{df['search_query'].iloc[i]}" for i in range(n_rows)]
    sample_df.index.name="RECIPE"
    # Fix 3: Drop the row (axis=0) - this removes the search_query row from the transposed df
    if 'search_query' in sample_df.index:
        sample_df = sample_df.drop('search_query', axis=0)

    # Fix 4: headers='keys' shows the column names (which are now your search queries)
    print(tabulate(sample_df,
                   headers='keys',
                   tablefmt='fancy_grid',
                   floatfmt='.2f',
                   stralign='left',
                   numalign='left',
                   showindex=True))

    print(f"\nDisplaying {n_rows} rows × {len(df.columns)} columns (transposed view)")

# Usage
display_wide_table_transposed(df, 3)

╒═════════════════╤════════════════════════════════════════════════╤══════════════════════════════════════════════════╤═════════════════════════════════════════════╕
│ RECIPE          │ Chicken breast, raw                            │ Chicken thigh, raw                               │ Turkey breast, raw                          │
╞═════════════════╪════════════════════════════════════════════════╪══════════════════════════════════════════════════╪═════════════════════════════════════════════╡
│ fdc_id          │ 2646170                                        │ 172855                                           │ 171098                                      │
├─────────────────┼────────────────────────────────────────────────┼──────────────────────────────────────────────────┼─────────────────────────────────────────────┤
│ food_name       │ Chicken, breast, boneless, skinless, raw       │ Chicken, skin (drumsticks and thighs), raw       │ Turkey, whole, breast, meat only, raw       │
├───

In [ ]:
# ==========================================
# DATA QUALITY CHECK
# ==========================================
print("\n" + "="*40)
print("NUTRITION DATA SUMMARY")
print("="*40)

# Basic info
print(f"\nShape: {df.shape[0]} foods × {df.shape[1]} columns")

# Duplicates
dup_count = df.duplicated().sum().item()
print(f"Duplicates: {dup_count} {'' if dup_count==0 else '⚠️'}")

# Missing values by column
print("\nMissing Values:")
for col in df.columns:
    null_count = df[col].isna().sum()
    if null_count > 0:
        print(f"   • {col}: {null_count} missing")
if df.isna().sum().sum() == 0:
    print("No missing values!")

# Quick stats for numeric columns
print("\nNutrient Stats (mean ± std):")
for col in ['calories', 'protein_g', 'carbs_g', 'fat_g']:
    if col in df.columns:
        mean_val = df[col].mean()
        std_val = df[col].std()
        print(f"   • {col}: {mean_val:.1f} ± {std_val:.1f}")

print("\n" + "="*40)
print("Data check complete!")
print("="*40)


NUTRITION DATA SUMMARY

Shape: 76 foods × 21 columns
Duplicates: 0 

Missing Values:
No missing values!

Nutrient Stats (mean ± std):
   • calories: 163.3 ± 177.5
   • protein_g: 11.3 ± 10.7
   • carbs_g: 20.6 ± 24.6
   • fat_g: 9.2 ± 18.1

Data check complete!


# fetch Structure exercise data from github repo

# fetch sample of data

In [ ]:
import requests
import pandas as pd

# 1. The Raw URL for the open-source Free Exercise Database
EXERCISE_DB_URL = "https://raw.githubusercontent.com/yuhonas/free-exercise-db/main/dist/exercises.json"

print("Fetching the global exercise database...")

# 2. Fetch the JSON data
response = requests.get(EXERCISE_DB_URL)

if response.status_code == 200:
    exercise_data = response.json()
    print(f"✅ Successfully downloaded {len(exercise_data)} exercises!\n")

    # 3. Let's look at the structure of a single exercise to verify our metadata!
    sample_exercise = exercise_data[0] # Grab the first one (usually Ab Crunches)

    print("--- SAMPLE EXERCISE STRUCTURE ---")
    print(f"Name: {sample_exercise.get('name')}")
    print(f"Primary Muscle: {sample_exercise.get('primaryMuscles')}")
    print(f"Equipment: {sample_exercise.get('equipment')}")
    print(f"Level: {sample_exercise.get('level')}")
    print(f"Mechanic: {sample_exercise.get('mechanic')}")

    # Instructions usually come as a list, we will need to join them into a paragraph
    instructions = sample_exercise.get('instructions', [])
    print(f"Instructions Steps: {len(instructions)}")

else:
    print(f"❌ Error fetching data: {response.status_code}")

Fetching the global exercise database...
✅ Successfully downloaded 873 exercises!

--- SAMPLE EXERCISE STRUCTURE ---
Name: 3/4 Sit-Up
Primary Muscle: ['abdominals']
Equipment: body only
Level: beginner
Mechanic: compound
Instructions Steps: 5


# fetch all data and store in df

In [ ]:
import pandas as pd

# Assuming you still have 'exercise_data' loaded from the previous cell
extracted_exercises = []

print(f"Enhancing and formatting {len(exercise_data)} exercises...\n")

for ex in exercise_data:
    # 1. Safely grab all the fields (handling missing data gracefully)
    name = ex.get('name', 'None')
    level = ex.get('level', 'None')
    mechanic = ex.get('mechanic', 'None')
    equipment = ex.get('equipment', 'None')
    force = ex.get('force', 'None')
    category = ex.get('category', 'None')

    # Extract lists and convert them to comma-separated strings
    primary_muscles = ", ".join(ex.get('primaryMuscles', []))
    secondary_muscles = ", ".join(ex.get('secondaryMuscles', []))

    # 2. Merge the instructions list into a single readable paragraph
    instructions_list = ex.get('instructions', [])
    merged_instructions = " ".join(instructions_list)

    # 3. Build the "Text Chunk" (This is what the AI will actually read!)
    text_chunk = (
        f"{name} is a {level} level {category} exercise targeting the {primary_muscles}. "
        f"It involves a {force} motion and is considered a {mechanic} movement. "
        f"Equipment needed: {equipment}. "
        f"Secondary muscles worked: {secondary_muscles}. "
        f"Instructions: {merged_instructions}"
    )

    # 4. Assemble the final Metadata dictionary
    metadata = {
        "exercise_name": name,
        "primary_muscles": primary_muscles,
        "secondary_muscles": secondary_muscles,
        "equipment": equipment,
        "level": level,
        "force": force,
        "category": category,
        "mechanic": mechanic,
        "text_chunk": text_chunk
    }

    extracted_exercises.append(metadata)

# 5. Save to CSV
df_exercises = pd.DataFrame(extracted_exercises)
df_exercises.to_csv('exercises_metadata.csv', index=False)

print("  ✅ Success! Formatted all instructions into readable AI text chunks.")
print("💾 Saved to 'exercises_metadata.csv'.")

Enhancing and formatting 873 exercises...

  ✅ Success! Formatted all instructions into readable AI text chunks.
💾 Saved to 'exercises_metadata.csv'.


In [ ]:
# ==========================================
# SIMPLE EXERCISE DATA CHECK
# ==========================================
print("\n" + "="*40)
print("💪 EXERCISE DATA SUMMARY")
print("="*40)

# Basic info
print(f"\n📈 Shape: {df_exercises.shape[0]} exercises × {df_exercises.shape[1]} columns")


# Missing values by column
print("\n🕳️ Missing Values:")
missing_total = 0
for col in df_exercises.columns:
    null_count = df_exercises[col].isna().sum()
    missing_total += null_count
    if null_count > 0:
        print(f"   • {col}: {null_count} missing")
if missing_total == 0:
    print("   ✅ No missing values in any column!")

# Duplicates check
dup_count = df_exercises.duplicated().sum()
print(f"\n🔍 Duplicate rows: {dup_count} {'✅' if dup_count==0 else '⚠️'}")

print("\n" + "="*40)
print("✅ Exercise data check complete!")
print("="*40)

# First 5 rows preview
print("\n👀 First 5 rows:")
df_exercises.head()


💪 EXERCISE DATA SUMMARY

📈 Shape: 873 exercises × 9 columns

🕳️ Missing Values:
   • equipment: 77 missing
   • force: 29 missing
   • mechanic: 87 missing

🔍 Duplicate rows: 0 ✅

✅ Exercise data check complete!

👀 First 5 rows:


,exercise_name,primary_muscles,secondary_muscles,equipment,level,force,category,mechanic,text_chunk
0,3/4 Sit-Up,abdominals,,body only,beginner,pull,strength,compound,3/4 Sit-Up is a beginner level strength exerci...
1,90/90 Hamstring,hamstrings,calves,body only,beginner,push,stretching,None,90/90 Hamstring is a beginner level stretching...
2,Ab Crunch Machine,abdominals,,machine,intermediate,pull,strength,isolation,Ab Crunch Machine is a intermediate level stre...
3,Ab Roller,abdominals,shoulders,other,intermediate,pull,strength,compound,Ab Roller is a intermediate level strength exe...
4,Adductor,adductors,,foam roll,intermediate,static,stretching,isolation,Adductor is a intermediate level stretching ex...


In [ ]:
import random
from tabulate import tabulate
import textwrap
random.seed(42)
# ==========================================
# RANDOM EXERCISE SAMPLE - DETAILED VIEW
# ==========================================
print("\n" + "🎲 RANDOM EXERCISE SAMPLE")
print("═" * 60)

# Get one random row
random_idx = random.randint(0, len(df_exercises)-1)
random_exercise = df_exercises.iloc[random_idx]

# Create formatted table
table_data = []
for col in df_exercises.columns:
    value = random_exercise[col]

    # Handle missing values
    if pd.isna(value):
        display_value = "⚠️  NOT PROVIDED"
    else:
        display_value = str(value).strip()
        # Wrap long text to multiple lines
        if len(display_value) > 40:
            display_value = textwrap.fill(display_value, width=40)

    table_data.append([col, display_value])

# Display with tabulate
print(f"\n📌 Selected Exercise #{random_idx}")
print(tabulate(table_data,
               headers=["Exercise Field", "Value"],
               tablefmt="fancy_grid",  # Beautiful grid
               colalign=("left", "left"),
               maxcolwidths=[20, 45]))  # Limit column widths

print("\n" + "═" * 60)


🎲 RANDOM EXERCISE SAMPLE
════════════════════════════════════════════════════════════

📌 Selected Exercise #654
╒═══════════════════╤═══════════════════════════════════════════════╕
│ Exercise Field    │ Value                                         │
╞═══════════════════╪═══════════════════════════════════════════════╡
│ exercise_name     │ See-Saw Press (Alternating Side Press)        │
├───────────────────┼───────────────────────────────────────────────┤
│ primary_muscles   │ shoulders                                     │
├───────────────────┼───────────────────────────────────────────────┤
│ secondary_muscles │ abdominals, triceps                           │
├───────────────────┼───────────────────────────────────────────────┤
│ equipment         │ dumbbell                                      │
├───────────────────┼───────────────────────────────────────────────┤
│ level             │ intermediate                                  │
├───────────────────┼──────────────────────────

# Use LLAMA model to impute the missing data

## split the missing data into 3 file to avoid token limit

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the raw data
df_raw = pd.read_csv('/content/exercises_metadata.csv').replace({np.nan: None})

# 2. Isolate the broken rows using our mask
REQUIRED_COLS = ['equipment', 'primary_muscles', 'secondary_muscles', 'level', 'force', 'category', 'mechanic']
mask = df_raw[REQUIRED_COLS].replace('', np.nan).isna().any(axis=1)

df_perfect = df_raw[~mask].copy()
df_needs_impute = df_raw[mask].copy()

# 3. Save the perfect data so it is safe and ready for later
df_perfect.to_csv('/content/perfect_data.csv', index=False)
print(f"✅ Saved {len(df_perfect)} perfect exercises to 'perfect_data.csv'")

# 4. Split the broken data into 3 equal chunks
# np.array_split handles the math automatically, even if it doesn't divide perfectly evenly
chunks = np.array_split(df_needs_impute, 3)

print("\nSplitting the remaining data into 3 worker files:")
for i, chunk in enumerate(chunks):
    filename = f'/content/missing_part_{i+1}.csv'
    chunk.to_csv(filename, index=False)
    print(f"  -> Saved {len(chunk)} exercises to '{filename}'")

✅ Saved 507 perfect exercises to 'perfect_data.csv'

Splitting the remaining data into 3 worker files:
  -> Saved 122 exercises to '/content/missing_part_1.csv'
  -> Saved 122 exercises to '/content/missing_part_2.csv'
  -> Saved 122 exercises to '/content/missing_part_3.csv'


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


# impute using llm model

In [ ]:
import pandas as pd
import numpy as np
import json
import time
from openai import OpenAI
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION (CHANGE THESE PER ACCOUNT!)
# ==========================================
GROQ_API_KEY = "##########################################"
TARGET_FILE = '/content/missing_part_1.csv'
OUTPUT_FILE = '/content/imputed_part_1.csv'

client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

# Load only the specific chunk assigned to this worker
df_worker = pd.read_csv(TARGET_FILE).replace({np.nan: None})
print(f"🚀 Worker Node started: Processing {len(df_worker)} exercises...\n")

# ==========================================
# 2. PROMPT & HELPER
# ==========================================
KINESIOLOGY_PROMPT = """
You are a Master Kinesiologist and an Expert Data Engineer building a production-grade fitness database.
You will be provided with a JSON object representing a single exercise. Some fields may be null, missing, or contradictory.

YOUR DIRECTIVES:
1. IMPUTATION: If 'equipment', 'primary_muscles', 'secondary_muscles', 'level', 'force', 'category', or 'mechanic' is null or empty, use your biomechanical expertise to fill in the correct value based on the 'exercise_name' and 'instructions'.
2. CORRECTION: If the 'equipment' listed contradicts the 'instructions', correct the equipment field.
3. STANDARDIZATION: Keep values lowercase. Use standard terms (e.g., 'barbell', 'dumbbell', 'cable', 'bodyweight', 'compound', 'isolation', 'push', 'pull').

You MUST return a strictly formatted JSON object containing ONLY the fully cleaned and imputed fields.
"""

def generate_text_chunk(data):
    return (
        f"{data.get('exercise_name', 'This')} is a {data.get('level', 'unknown')} level {data.get('category', 'unknown')} exercise targeting the {data.get('primary_muscles', 'unknown')}. "
        f"It involves a {data.get('force', 'unknown')} motion and is considered a {data.get('mechanic', 'unknown')} movement. "
        f"Equipment needed: {data.get('equipment', 'unknown')}. "
        f"Secondary muscles worked: {data.get('secondary_muscles', 'unknown')}. "
        f"Instructions: {data.get('instructions', '')}"
    )

# ==========================================
# 3. THE IMPUTATION LOOP
# ==========================================
recovered_exercises = []

with tqdm(total=len(df_worker), desc="Imputing", unit="ex", colour="blue") as pbar:
    for index, row in df_worker.iterrows():
        row_dict = row.to_dict()

        # 3 Retries per row
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    response_format={ "type": "json_object" },
                    temperature=0.1,
                    messages=[
                        {"role": "system", "content": KINESIOLOGY_PROMPT},
                        {"role": "user", "content": json.dumps(row_dict)}
                    ]
                )

                cleaned_metadata = json.loads(response.choices[0].message.content)
                cleaned_metadata['instructions'] = row_dict.get('instructions', '')
                if 'exercise_name' not in cleaned_metadata:
                    cleaned_metadata['exercise_name'] = row_dict.get('exercise_name', 'Unknown')

                cleaned_metadata['text_chunk'] = generate_text_chunk(cleaned_metadata)
                recovered_exercises.append(cleaned_metadata)

                time.sleep(1.5) # Gentle wait
                break

            except Exception as e:
                if "429" in str(e).lower() or "rate" in str(e).lower():
                    time.sleep(5) # Backoff if rate limited
                else:
                    break # Skip on bad JSON

        pbar.update(1)

# ==========================================
# 4. SAVE THE WORKER'S OUTPUT
# ==========================================
df_output = pd.DataFrame(recovered_exercises)
df_output.to_csv(OUTPUT_FILE, index=False)
print("\n" + "="*50)
print(f"WORKER FINISHED! Saved {len(df_output)} imputed rows to {OUTPUT_FILE}")
print("="*50)

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the raw dataset
df_raw = pd.read_csv('/content/exercises_metadata.csv')

# 2. Define the exact columns that must be fully populated
REQUIRED_COLS = ['equipment', 'primary_muscles', 'secondary_muscles', 'level', 'force', 'category', 'mechanic']

# 3. Create the mask to find missing values (treating empty strings as NaN)
mask_missing = df_raw[REQUIRED_COLS].replace('', np.nan).isna().any(axis=1)

# 4. Isolate the perfect, non-null rows using the inverse of the mask (~)
df_perfect = df_raw[~mask_missing].copy()

# 5. Verify the split
print(f"Total exercises loaded: {len(df_raw)}")
print(f" Perfect exercises (non-null): {len(df_perfect)}")
print(f" Broken exercises (null): {len(df_raw) - len(df_perfect)}")

# Show the first 3 rows to confirm it looks good
display(df_perfect.head(3))

Total exercises loaded: 873
 Perfect exercises (non-null): 507
 Broken exercises (null): 366


,exercise_name,primary_muscles,secondary_muscles,equipment,level,force,category,mechanic,text_chunk
3,Ab Roller,abdominals,shoulders,other,intermediate,pull,strength,compound,Ab Roller is a intermediate level strength exe...
6,Advanced Kettlebell Windmill,abdominals,"glutes, hamstrings, shoulders",kettlebells,intermediate,push,strength,isolation,Advanced Kettlebell Windmill is a intermediate...
9,Alternate Hammer Curl,biceps,forearms,dumbbell,beginner,pull,strength,isolation,Alternate Hammer Curl is a beginner level stre...


In [ ]:
df_imputed1=pd.read_csv('/content/imputed_part_1.csv')
df_imputed2=pd.read_csv('/content/imputed_part_2.csv')
df_imputed3=pd.read_csv('/content/imputed_part_3.csv')

df_final_impute_concat=pd.concat(objs=[df_perfect,df_imputed1,df_imputed2,df_imputed3],ignore_index=True)
df_final_impute_concat.to_csv('/content/final_merged_imputed_exercice.csv',index=False)
df_final_impute_concat.drop(columns="instructions",inplace=True)
print(f"Successfully merged! Total rows: {len(df_final_impute_concat)}")

Successfully merged! Total rows: 873


In [ ]:
from IPython.display import display, HTML

# 1. Convert the list of cleaned exercises into a new DataFrame
df_imputed = pd.DataFrame(df_final_impute_concat)

# 2. Select a random sample (e.g., 5 rows) to inspect the AI's work
# We use .sample() so you see different exercises every time you run it
sample_df = df_imputed.sample(min(5, len(df_imputed)))

# 3. Create a clean, formatted display
print("============================================================")
print("RANDOM LLM IMPUTATION SAMPLE")
print("============================================================")

# We display only the metadata columns to keep the table readable
cols_to_show = ['exercise_name', 'primary_muscles', 'equipment', 'level', 'force', 'mechanic']

# Apply some basic styling for better readability in Colab
styled_sample = sample_df[cols_to_show].style.set_properties(**{
    'background-color': '#f9f9f9',
    'color': 'black',
    'border-color': 'white'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4CAF50'), ('color', 'white')]}
])

display(styled_sample)

# 4. Detailed View of the first sample's text_chunk
# This lets you see how the metadata was merged into the paragraph for the RAG system
print("\n📝  DETAILED TEXT CHUNK (AI Paragraph Example):")
print("-" * 60)
print(sample_df.iloc[0]['text_chunk'])
print("-" * 60)

RANDOM LLM IMPUTATION SAMPLE


,exercise_name,primary_muscles,equipment,level,force,mechanic
49,Bench Jump,quadriceps,body only,intermediate,push,compound
44,Barbell Walking Lunge,quadriceps,barbell,beginner,push,compound
470,Suspended Split Squat,quadriceps,other,intermediate,push,compound
872,yoke walk,quadriceps,yoke,intermediate,concentric,compound
63,Board Press,triceps,barbell,intermediate,push,compound



📝  DETAILED TEXT CHUNK (AI Paragraph Example):
------------------------------------------------------------
Bench Jump is a intermediate level plyometrics exercise targeting the quadriceps. It involves a push motion and is considered a compound movement. Equipment needed: body only. Secondary muscles worked: calves, glutes, hamstrings. Instructions: Begin with a box or bench 1-2 feet in front of you. Stand with your feet shoulder width apart. This will be your starting position. Perform a short squat in preparation for the jump; swing your arms behind you. Rebound out of this position, extending through the hips, knees, and ankles to jump as high as possible. Swing your arms forward and up. Jump over the bench, landing with the knees bent, absorbing the impact through the legs. Turn around and face the opposite direction, then jump back over the bench.
------------------------------------------------------------


In [ ]:
# Missing values by column
print("\n🕳️ Missing Values:")
missing_total = 0
for col in df_imputed.columns:
    null_count = df_imputed[col].isna().sum()
    missing_total += null_count
    if null_count > 0:
        print(f"   • {col}: {null_count} missing")
if missing_total == 0:
    print("   ✅ No missing values in any column!")


🕳️ Missing Values:
   • secondary_muscles: 1 missing


# Ensure the llm model dont change the NON NULL data

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the Data
df_old = pd.read_csv('/content/exercises_metadata.csv')
df_new = pd.read_csv('/content/final_merged_imputed_exercice.csv')

# 2. Merge them on 'exercise_name' to ensure perfect alignment
# 'suffixes' will help us distinguish columns (e.g., equipment_old vs equipment_new)
df_audit = pd.merge(
    df_old,
    df_new,
    on='exercise_name',
    suffixes=('_old', '_new')
)

metadata_cols = ['equipment', 'force', 'mechanic', 'level', 'primary_muscles']
audit_results = []

for col in metadata_cols:
    old_col = f"{col}_old"
    new_col = f"{col}_new"

    # Ensure both are strings to avoid type-mismatch errors during comparison
    df_audit[old_col] = df_audit[old_col].astype(str).replace('nan', np.nan)
    df_audit[new_col] = df_audit[new_col].astype(str).replace('nan', np.nan)

    # A. Find IMPUTATIONS: Was null in old, has value in new
    imputed_mask = df_audit[old_col].isna() & df_audit[new_col].notna()
    imputed_df = df_audit[imputed_mask][['exercise_name', new_col]].copy()
    imputed_df['action'] = '✨ IMPUTED'
    imputed_df['original'] = 'NaN'
    imputed_df['column'] = col
    imputed_df.rename(columns={new_col: 'new_value'}, inplace=True)

    # B. Find OVERWRITES: Was NOT null in old, and new is DIFFERENT
    # We use .ne() (Not Equal) which handles NaNs more gracefully than !=
    violation_mask = df_audit[old_col].notna() & (df_audit[old_col] != df_audit[new_col])
    violation_df = df_audit[violation_mask][['exercise_name', new_col, old_col]].copy()
    violation_df['action'] = '🚨 OVERWRITE'
    violation_df['column'] = col
    violation_df.rename(columns={new_col: 'new_value', old_col: 'original'}, inplace=True)

    audit_results.extend([imputed_df, violation_df])

# 3. Combine results
if audit_results:
    full_audit = pd.concat(audit_results).reset_index(drop=True)
else:
    full_audit = pd.DataFrame(columns=['exercise_name', 'new_value', 'action', 'original', 'column'])

# ==========================================
# 📊 FINAL VERIFICATION REPORT
# ==========================================
print("========================================")
print("🛡️  LLM IMPUTATION AUDIT REPORT")
print("========================================")
print(f"Total Imputations: {len(full_audit[full_audit['action'] == '✨ IMPUTED'])}")
print(f"Total Overwrites:  {len(full_audit[full_audit['action'] == '🚨 OVERWRITE'])}")
print("----------------------------------------")

if not full_audit[full_audit['action'] == '🚨 OVERWRITE'].empty:
    print("\n⚠️  WARNING: The model changed existing data in these rows:")
    # Display the first few so you can check if the change was actually a "smart fix"
    display(full_audit[full_audit['action'] == '🚨 OVERWRITE'].head(10))

    # APPLY SAFETY LOCK:
    # This ensures your FINAL df has the original values wherever they existed,
    # and only uses Llama's data for the holes (NaNs).
    df_final = df_old.set_index('exercise_name').combine_first(df_new.set_index('exercise_name')).reset_index()
    print("\n✅ Safety Lock Applied: Original values preserved, NaNs filled by Llama.")
else:
    df_final = df_new
    print("\n✅ Clean Run: No original data was ruined!")

# Save the absolute final verified version
df_final.to_csv('exercises_final_verified.csv', index=False)

🛡️  LLM IMPUTATION AUDIT REPORT
Total Imputations: 0
Total Overwrites:  1
----------------------------------------

⚠️  WARNING: The model changed existing data in these rows:


,exercise_name,new_value,action,original,column
0,Donkey Calf Raises,machine,🚨 OVERWRITE,other,equipment



✅ Safety Lock Applied: Original values preserved, NaNs filled by Llama.


## return the old data for overited row

In [ ]:
mask=df_new["exercise_name"]=="Donkey Calf Raises"
df_new.loc[mask,"equipment"]="other"


print(df_new.loc[mask, "equipment"])

589    other
Name: equipment, dtype: object


In [ ]:
import pandas as pd
import numpy as np


# 2. Merge them on 'exercise_name' to ensure perfect alignment
# 'suffixes' will help us distinguish columns (e.g., equipment_old vs equipment_new)
df_audit = pd.merge(
    df_old,
    df_new,
    on='exercise_name',
    suffixes=('_old', '_new')
)

metadata_cols = ['equipment', 'force', 'mechanic', 'level', 'primary_muscles']
audit_results = []

for col in metadata_cols:
    old_col = f"{col}_old"
    new_col = f"{col}_new"

    # Ensure both are strings to avoid type-mismatch errors during comparison
    df_audit[old_col] = df_audit[old_col].astype(str).replace('nan', np.nan)
    df_audit[new_col] = df_audit[new_col].astype(str).replace('nan', np.nan)

    # A. Find IMPUTATIONS: Was null in old, has value in new
    imputed_mask = df_audit[old_col].isna() & df_audit[new_col].notna()
    imputed_df = df_audit[imputed_mask][['exercise_name', new_col]].copy()
    imputed_df['action'] = '✨ IMPUTED'
    imputed_df['original'] = 'NaN'
    imputed_df['column'] = col
    imputed_df.rename(columns={new_col: 'new_value'}, inplace=True)

    # B. Find OVERWRITES: Was NOT null in old, and new is DIFFERENT
    # We use .ne() (Not Equal) which handles NaNs more gracefully than !=
    violation_mask = df_audit[old_col].notna() & (df_audit[old_col] != df_audit[new_col])
    violation_df = df_audit[violation_mask][['exercise_name', new_col, old_col]].copy()
    violation_df['action'] = '🚨 OVERWRITE'
    violation_df['column'] = col
    violation_df.rename(columns={new_col: 'new_value', old_col: 'original'}, inplace=True)

    audit_results.extend([imputed_df, violation_df])

# 3. Combine results
if audit_results:
    full_audit = pd.concat(audit_results).reset_index(drop=True)
else:
    full_audit = pd.DataFrame(columns=['exercise_name', 'new_value', 'action', 'original', 'column'])

# ==========================================
# 📊 FINAL VERIFICATION REPORT
# ==========================================
print("========================================")
print("🛡️  LLM IMPUTATION AUDIT REPORT")
print("========================================")
print(f"Total Imputations: {len(full_audit[full_audit['action'] == '✨ IMPUTED'])}")
print(f"Total Overwrites:  {len(full_audit[full_audit['action'] == '🚨 OVERWRITE'])}")
print("----------------------------------------")

if not full_audit[full_audit['action'] == '🚨 OVERWRITE'].empty:
    print("\n⚠️  WARNING: The model changed existing data in these rows:")
    # Display the first few so you can check if the change was actually a "smart fix"
    display(full_audit[full_audit['action'] == '🚨 OVERWRITE'].head(10))

    # APPLY SAFETY LOCK:
    # This ensures your FINAL df has the original values wherever they existed,
    # and only uses Llama's data for the holes (NaNs).
    df_final = df_old.set_index('exercise_name').combine_first(df_new.set_index('exercise_name')).reset_index()
    print("\n✅ Safety Lock Applied: Original values preserved, NaNs filled by Llama.")
else:
    df_final = df_new
    print("\n✅ Clean Run: No original data was ruined!")

# Save the absolute final verified version
df_final.to_csv('exercises_final_verified.csv', index=False)

🛡️  LLM IMPUTATION AUDIT REPORT
Total Imputations: 0
Total Overwrites:  0
----------------------------------------

✅ Clean Run: No original data was ruined!


# store data CSV on drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

print("\n" + "="*50)
print("COPYING FILES TO GOOGLE DRIVE")
print("="*50)

# Files to copy
files_to_copy = [
    "/content/exercises_final_verified.csv"
]

destination = '/content/drive/MyDrive/SPORT_METADATA/Structure data/Gym_exercises'


# Copy each file
for file_path in files_to_copy:
    file_name = os.path.basename(file_path)
    print(f"\n📄 Copying: {file_name}...", end=" ")

    if os.path.exists(file_path):
        shutil.copy2(file_path, destination)
        print("✅ Done")
    else:
        print("❌ File not found")

print("\n" + "="*50)
print("✅ All files copied to:", destination)
print("="*50)


COPYING FILES TO GOOGLE DRIVE

📄 Copying: exercises_final_verified.csv... ✅ Done

✅ All files copied to: /content/drive/MyDrive/SPORT_METADATA/Structure data/Gym_exercises
